In [ ]:
# ============================================================================
# CELL 1 — Setup: load splits.npz and metadata for verification
# ============================================================================
#
# Goal of this notebook: prove that the existing strategy_A splits.npz has
# zero collector overlap across train/val/test, reproduce the imbalance
# score from the original split run, and document the partition as the
# baseline that strategy_B will inherit.
#
# This cell just loads things and prints shapes. No assertions yet.

from pathlib import Path
import json
import numpy as np
import pandas as pd

# --- Path resolution (same find_root pattern as the rest of the project) ---
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.json').is_file() and (p / 'data').is_dir():
            return p
    raise RuntimeError("project root (config.json + data/) not found above cwd")

ROOT = find_root()
with open(ROOT / 'config.json') as f:
    CONFIG = json.load(f)
P = CONFIG['paths']

# --- Load the existing splits.npz ---
SPLITS_PATH = ROOT / P['strategy_a_artifacts'] / 'splits.npz'
splits = dict(np.load(SPLITS_PATH, allow_pickle=True))

# --- Load metadata for the file → collector_id mapping (used in later cells) ---
metadata = pd.read_csv(ROOT / P['metadata_csv'])

# --- Sanity prints ---
print(f"splits.npz: {SPLITS_PATH}")
print(f"  size: {SPLITS_PATH.stat().st_size / 1e6:.1f} MB")
print(f"  keys: {sorted(splits.keys())}\n")

for s in ['train', 'val', 'test']:
    X = splits[f'X_{s}']
    Y = splits[f'Y_{s}']
    idx = splits[f'{s}_idx']
    print(f"  {s:5s}: X={X.shape} {X.dtype}  Y={Y.shape} {Y.dtype}  "
          f"idx range=[{idx.min()}, {idx.max()}]  n_idx={len(idx)}")

n_total = sum(splits[f'X_{s}'].shape[0] for s in ['train', 'val', 'test'])
print(f"\n  total segments: {n_total}  (expected 168,239 from memory)")

print(f"\nmetadata.csv: {len(metadata)} files, "
      f"{metadata['collector_id'].nunique()} unique collectors  "
      f"(expected 3,656 files / 408 collectors)")

print(f"\nfeature space: {splits['X_train'].shape[1]} dims, "
      f"{len(splits['feature_names'])} feature group names "
      f"(expected 960 dims / 56 groups)")

print(f"\nclass_names ({len(splits['class_names'])}):")
for i, c in enumerate(splits['class_names']):
    print(f"  {i:2d}  {c}")

splits.npz: /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/splits.npz
  size: 533.8 MB
  keys: ['X_test', 'X_train', 'X_val', 'Y_test', 'Y_train', 'Y_val', 'class_names', 'feature_names', 'scaler_mean', 'scaler_scale', 'test_idx', 'train_idx', 'val_idx']

  train: X=(100166, 960) float32  Y=(100166, 15) int8  idx range=[0, 168238]  n_idx=100166
  val  : X=(34245, 960) float32  Y=(34245, 15) int8  idx range=[568, 167631]  n_idx=34245
  test : X=(33828, 960) float32  Y=(33828, 15) int8  idx range=[2875, 166883]  n_idx=33828

  total segments: 168239  (expected 168,239 from memory)

metadata.csv: 3656 files, 408 unique collectors  (expected 3,656 files / 408 collectors)

feature space: 960 dims, 56 feature group names (expected 960 dims / 56 groups)

class_names (15):
   0  bell_ringing
   1  coffee_machine
   2  cutlery_dishes
   3  door_open_close
   4  footsteps
   5  keyboard_typing
   6  keychain
   7  light_switch
   8  microwave
   9  phone_ringing
  10  running_wa

In [2]:
# ============================================================================
# CELL 2 — Reconstruct the segment → file_id → collector_id mapping
# ============================================================================
#
# splits.npz stores global integer indices (train_idx, val_idx, test_idx) but
# NOT the segment → file mapping that those indices refer to. The original
# data_preparation.ipynb built a per-segment collector_all array of length
# 168,239, but it wasn't persisted. We rebuild it here by replicating the
# exact iteration order from the source:
#
#   npz_files = sorted(FEATURES_DIR.glob("*.npz"))   # alphabetical
#   for each file: T segments = data["annotations"].shape[0]
#
# As long as we use the same sort and the same segment-count rule, our
# segment→collector mapping will match the one that produced splits.npz.

import time
from tqdm import tqdm

FEATURES_DIR = ROOT / P['orig_npz']     # data/audio_features/

# --- 2.1  Build file_id → collector_id lookup (same logic as data_prep cell 2) ---
metadata['file_id'] = metadata['filename'].str.replace(r'\.[^.]+$', '', regex=True)
file_to_collector = dict(zip(metadata['file_id'], metadata['collector_id']))
print(f"Built lookup: {len(file_to_collector)} files → collectors")

# --- 2.2  Walk all npz files in the same sorted() order data_prep used ---
npz_files = sorted(FEATURES_DIR.glob('*.npz'))
print(f"Found {len(npz_files)} npz files in features dir")
assert len(npz_files) == len(metadata), \
    f"npz count {len(npz_files)} ≠ metadata count {len(metadata)}"

# --- 2.3  For each file, read segment count (T = annotations.shape[0]) ---
# We only load the 'annotations' array per file, not features → fast.
t0 = time.time()
file_ids = []
segment_counts = []
unknown_collectors = 0

for npz_path in tqdm(npz_files, desc='counting segments'):
    file_id = npz_path.stem
    # mmap_mode='r' avoids loading whole file just to get a shape
    with np.load(npz_path, allow_pickle=True, mmap_mode='r') as d:
        T = d['annotations'].shape[0]
    file_ids.append(file_id)
    segment_counts.append(T)
    if file_id not in file_to_collector:
        unknown_collectors += 1

print(f"\nWalked {len(npz_files)} files in {time.time()-t0:.1f}s")
print(f"Files with no collector mapping: {unknown_collectors}  (expect 0)")

# --- 2.4  Expand into per-segment arrays of length N_total ---
# segment_file_id[i]    = file_id of segment i
# segment_collector[i]  = collector_id of segment i
segment_file_id   = np.repeat(file_ids,       segment_counts)
segment_collector = np.repeat(
    [file_to_collector[fid] for fid in file_ids],
    segment_counts
)

N_total = len(segment_file_id)
print(f"\nReconstructed segment-level arrays:")
print(f"  segment_file_id:    shape={segment_file_id.shape}, dtype={segment_file_id.dtype}")
print(f"  segment_collector:  shape={segment_collector.shape}, dtype={segment_collector.dtype}")

# --- 2.5  Sanity checks ---
expected_total = splits['X_train'].shape[0] + splits['X_val'].shape[0] + splits['X_test'].shape[0]
assert N_total == expected_total, \
    f"Reconstruction mismatch: got {N_total} segments, splits cover {expected_total}"
assert N_total == 168_239, f"Total {N_total} ≠ 168,239 from memory"

# Every global index used by the splits must be in [0, N_total)
all_idx = np.concatenate([splits['train_idx'], splits['val_idx'], splits['test_idx']])
assert all_idx.min() == 0 and all_idx.max() == N_total - 1, \
    f"Index range [{all_idx.min()}, {all_idx.max()}] doesn't span [0, {N_total-1}]"
assert len(np.unique(all_idx)) == N_total, \
    f"Indices don't partition: {len(np.unique(all_idx))} unique vs {N_total} total " \
    "(some segment is missing or duplicated)"

print(f"\n  PASS  total segments match: {N_total} = 168,239")
print(f"  PASS  train/val/test indices form a partition of [0, {N_total-1}]")

# --- 2.6  Per-file segment count distribution (for the report later) ---
seg_counts = np.array(segment_counts)
print(f"\nSegments per file: min={seg_counts.min()}, max={seg_counts.max()}, "
      f"median={int(np.median(seg_counts))}, mean={seg_counts.mean():.1f}")
print(f"Total collectors in dataset: {len(set(segment_collector))} "
      f"(expect 408 from metadata)")

Built lookup: 3656 files → collectors
Found 3656 npz files in features dir


counting segments: 100%|██████████| 3656/3656 [00:02<00:00, 1539.53it/s]



Walked 3656 files in 2.4s
Files with no collector mapping: 0  (expect 0)

Reconstructed segment-level arrays:
  segment_file_id:    shape=(168239,), dtype=<U6
  segment_collector:  shape=(168239,), dtype=<U16

  PASS  total segments match: 168239 = 168,239
  PASS  train/val/test indices form a partition of [0, 168238]

Segments per file: min=23, max=78, median=44, mean=46.0
Total collectors in dataset: 408 (expect 408 from metadata)


In [3]:
# ============================================================================
# CELL 3 — Verify the split: zero collector overlap + reproduce balance score
# ============================================================================
#
# This is the actual leakage assertion. If train/val/test share even one
# collector, the split is broken — any reported test metric becomes a
# measurement of "model learned this person's home", not "model generalizes
# across homes". With the segment→collector mapping from Cell 2, the check
# is one set intersection per pair.

# --- 3.1  Extract the collector set for each partition --------------------
train_collectors = set(segment_collector[splits['train_idx']])
val_collectors   = set(segment_collector[splits['val_idx']])
test_collectors  = set(segment_collector[splits['test_idx']])

overlap_tv = train_collectors & val_collectors
overlap_tt = train_collectors & test_collectors
overlap_vt = val_collectors   & test_collectors

print("=== COLLECTOR OVERLAP CHECK ===")
print(f"  train ∩ val   = {len(overlap_tv)}  (expect 0)")
print(f"  train ∩ test  = {len(overlap_tt)}  (expect 0)")
print(f"  val   ∩ test  = {len(overlap_vt)}  (expect 0)")

# The hard assertion: if any of these fire, the split is leaky and
# everything downstream is suspect.
assert not overlap_tv, f"LEAK: train∩val has {len(overlap_tv)} collectors: {overlap_tv}"
assert not overlap_tt, f"LEAK: train∩test has {len(overlap_tt)} collectors: {overlap_tt}"
assert not overlap_vt, f"LEAK: val∩test has {len(overlap_vt)} collectors: {overlap_vt}"
print("  PASS  zero collector overlap across all three partitions\n")

# --- 3.2  Collector partition counts --------------------------------------
n_tr_c = len(train_collectors)
n_va_c = len(val_collectors)
n_te_c = len(test_collectors)
n_total_c = len(train_collectors | val_collectors | test_collectors)

print("=== COLLECTOR PARTITION ===")
print(f"  train: {n_tr_c:>3d}  ({n_tr_c/n_total_c:.1%})")
print(f"  val:   {n_va_c:>3d}  ({n_va_c/n_total_c:.1%})")
print(f"  test:  {n_te_c:>3d}  ({n_te_c/n_total_c:.1%})")
print(f"  total: {n_total_c:>3d}  (expect 408)")
print(f"  memory expectation: 244/82/82 → check above\n")

assert n_total_c == 408, f"Got {n_total_c} collectors, expected 408"

# --- 3.3  Segment partition stats -----------------------------------------
n_tr_s = len(splits['train_idx'])
n_va_s = len(splits['val_idx'])
n_te_s = len(splits['test_idx'])
N_s = n_tr_s + n_va_s + n_te_s

print("=== SEGMENT PARTITION ===")
print(f"  train: {n_tr_s:>6d}  ({n_tr_s/N_s:.1%})")
print(f"  val:   {n_va_s:>6d}  ({n_va_s/N_s:.1%})")
print(f"  test:  {n_te_s:>6d}  ({n_te_s/N_s:.1%})")
print(f"  total: {N_s:>6d}\n")

# --- 3.4  Per-class positive rate per split -------------------------------
class_names = list(splits['class_names'])
p_tr = splits['Y_train'].mean(axis=0)
p_va = splits['Y_val'].mean(axis=0)
p_te = splits['Y_test'].mean(axis=0)

# Format as a quick table
print("=== PER-CLASS POSITIVE RATE PER SPLIT (%) ===")
print(f"  {'class':<30s}  {'train':>7s}  {'val':>7s}  {'test':>7s}  {'|Δmax|':>7s}")
for i, c in enumerate(class_names):
    dmax = max(abs(p_tr[i] - p_va[i]), abs(p_tr[i] - p_te[i]), abs(p_va[i] - p_te[i]))
    print(f"  {c:<30s}  {p_tr[i]*100:>6.2f}%  {p_va[i]*100:>6.2f}%  {p_te[i]*100:>6.2f}%  {dmax*100:>6.2f}%")

# --- 3.5  KL imbalance score ----------------------------------------------
# Multi-label setting: each class is a Bernoulli, so KL between splits is
# the sum over classes of D_KL(Bern(p_split_a) || Bern(p_split_b)).
# Several common aggregations — print all so we can match the memory's 0.0054.

def bernoulli_kl(p, q, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    q = np.clip(q, eps, 1 - eps)
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))

kl_tv = bernoulli_kl(p_tr, p_va).sum()
kl_tt = bernoulli_kl(p_tr, p_te).sum()
kl_vt = bernoulli_kl(p_va, p_te).sum()

print(f"\n=== KL IMBALANCE SCORES (multi-label Bernoulli, nats) ===")
print(f"  KL(train || val)  = {kl_tv:.4f}")
print(f"  KL(train || test) = {kl_tt:.4f}")
print(f"  KL(val   || test) = {kl_vt:.4f}")
print(f"  mean of three     = {(kl_tv + kl_tt + kl_vt)/3:.4f}")
print(f"  mean(t||v, t||t)  = {(kl_tv + kl_tt)/2:.4f}")
print(f"  sum(t||v, t||t)   = {kl_tv + kl_tt:.4f}")
print(f"  memory expectation: 0.0054  → check which formula matches\n")

# --- 3.6  Save the segment→collector mapping for Strategy B ---------------
# Strategy B needs to know the partition each collector belongs to so that
# augmented files (which inherit their source's collector_id) land in the
# correct partition. Save the per-collector mapping as a small CSV.
collector_to_split = {}
for c in train_collectors: collector_to_split[c] = 'train'
for c in val_collectors:   collector_to_split[c] = 'val'
for c in test_collectors:  collector_to_split[c] = 'test'

import pandas as pd
collector_df = pd.DataFrame({
    'collector_id': list(collector_to_split.keys()),
    'split':        list(collector_to_split.values()),
}).sort_values(['split', 'collector_id']).reset_index(drop=True)

OUT_CSV = ROOT / P['strategy_a_artifacts'] / 'collector_partition.csv'
collector_df.to_csv(OUT_CSV, index=False)
print(f"Saved collector→split mapping: {OUT_CSV}")
print(f"  rows: {len(collector_df)}  ({collector_df['split'].value_counts().to_dict()})")

=== COLLECTOR OVERLAP CHECK ===
  train ∩ val   = 0  (expect 0)
  train ∩ test  = 0  (expect 0)
  val   ∩ test  = 0  (expect 0)
  PASS  zero collector overlap across all three partitions

=== COLLECTOR PARTITION ===
  train: 244  (59.8%)
  val:    82  (20.1%)
  test:   82  (20.1%)
  total: 408  (expect 408)
  memory expectation: 244/82/82 → check above

=== SEGMENT PARTITION ===
  train: 100166  (59.5%)
  val:    34245  (20.4%)
  test:   33828  (20.1%)
  total: 168239

=== PER-CLASS POSITIVE RATE PER SPLIT (%) ===
  class                             train      val     test   |Δmax|
  bell_ringing                      1.09%    1.40%    1.43%    0.34%
  coffee_machine                    3.78%    3.13%    3.65%    0.65%
  cutlery_dishes                    6.03%    5.34%    6.05%    0.71%
  door_open_close                   3.57%    3.44%    3.18%    0.39%
  footsteps                        11.41%   11.78%   12.58%    1.17%
  keyboard_typing                   9.11%    9.07%    9.85%    0.7